# 22 · Higher Secondary batches per school — HSCAP 2026 (Kerala)

**Why:** the capacity of a Higher Secondary school is set by its **batches**: **50 seats per batch** (Government
Order G.O.(Rt) No. 5293/2026/GEDN, prospectus erratum: "ബാച്ചൊന്നിന് 50 മെറിറ്റ് സീറ്റുകൾ" — 50 merit seats per
batch). The number of batches per school was the last missing piece of school capacity.

**Source:** HSCAP — Higher Secondary Centralised Allotment Process, Directorate of General Education (Higher
Secondary Wing), Government of Kerala, `https://hscap.kerala.gov.in/` → **School List** (public; one page per
district, the page's own district menu). For each school it lists the HSS code, management type, local body and the
courses offered; the course cell's tooltip states the **batches per course**, e.g.
`1 - Physics, Chemistry, Biology,Mathematics [3 Batches]`.

**Link to our schools:** the HSS code (Sametham school pages list it; `sam_hss_code` in the capacity file).

**Funding per batch:** some schools run batches of two funding types (e.g. Aided + Un-Aided self-financing);
the tooltip groups them under a heading, kept as `batch_funding`.

**Checks:** courses in the tooltip = courses listed in the cell (asserted for every school); school counts per
district compared with Sametham's own consolidation of HSE schools.

**Output:** raw pages `data/raw/hscap/school_list_district_<NN>_<date>.html`; one row per school and course with
its batches → `data/interim/schools/hscap_hss_courses.csv`. Notebook 09 adds the batches and seats to
`FINAL COMPLETE DATASET/schools/kerala_schools_capacity.csv`. 15 polite requests (2 s apart); pages already saved are
not downloaded again.

## Project folder (laptop or Colab)

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

## Imports and settings

In [3]:
import datetime as dt
import html
import re
import time

import pandas as pd
import requests
import urllib3.util.connection

urllib3.util.connection.HAS_IPV6 = False  # IPv6 times out from this network (see notebook 14)
RAW = PROJECT / "data" / "raw" / "hscap"
OUT = PROJECT / "data" / "interim" / "schools" / "hscap_hss_courses.csv"
URL = "https://hscap.kerala.gov.in/school.php"
SAMETHAM_HSE = "https://sametham.kite.kerala.gov.in/publicView/schoolconsolidationhse"
DISTRICTS = {1: "THIRUVANANTHAPURAM", 2: "KOLLAM", 3: "PATHANAMTHITTA", 4: "ALAPPUZHA", 5: "KOTTAYAM", 6: "IDUKKY",
             7: "ERNAKULAM", 8: "THRISSUR", 9: "PALAKKAD", 10: "KOZHIKKODE", 11: "MALAPPURAM", 12: "WAYANAD",
             13: "KANNUR", 14: "KASARGODE", 15: "OTHERS"}  # the district menu of the HSCAP "School List" page
HEADERS = {"User-Agent": "msc-spatial-inequality-study/0.1 (academic project)"}
DELAY_S = 2

### `fetch()`

In [4]:
# Chain of the site certificate: hscap.kerala.gov.in -> Let's Encrypt "YR2" -> "ISRG Root YR" (Let's Encrypt's new
# root, not yet in certifi) -> ISRG Root X1 (in certifi). The server sends only its own certificate; the two
# missing certificates are published at the "CA Issuers" addresses written inside the certificates themselves.
LETSENCRYPT_ISSUERS = ["http://yr2.i.lencr.org/", "http://yr.i.lencr.org/"]


def ca_bundle():
    """certifi's trusted roots + the two missing certificates. Verification stays on: the chain must still end at
    ISRG Root X1, which certifi trusts, so a wrong certificate would be rejected."""
    import ssl
    import certifi
    f = RAW / "ca_bundle_with_letsencrypt_chain.pem"
    if not f.exists():
        extra = [ssl.DER_cert_to_PEM_cert(requests.get(u, headers=HEADERS, timeout=60).content) for u in LETSENCRYPT_ISSUERS]
        f.write_text(open(certifi.where(), encoding="utf-8").read() + "\n" + "\n".join(extra), encoding="utf-8")
    return str(f)


def fetch(d, session):
    """School list of one district (the page's own form: POST cmbdist=<d>); saved once, reused on later runs."""
    saved = sorted(RAW.glob(f"school_list_district_{d:02d}_*.html"))
    if saved:
        return saved[-1].read_text(encoding="utf-8")
    time.sleep(DELAY_S)
    r = session.post(URL, data={"cmbdist": d}, timeout=120, verify=CA)
    r.raise_for_status()
    f = RAW / f"school_list_district_{d:02d}_{dt.date.today().isoformat()}.html"
    f.write_text(r.text, encoding="utf-8")
    return r.text

### `parse()`

In [5]:
COURSE = re.compile(r"(\d+)\s*-\s*(.*?)\s*\[(\d+)\s*Batch(?:es)?\]")


def text(cell):
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", cell))).strip()


def tooltip_courses(title, default_funding):
    """Tooltip lines 'code - subjects [N Batches]'. A school may run batches of two funding types (e.g. Aided and
    Un-Aided = self-financing); those lines then follow a heading line such as 'Aided' or 'Un-Aided'."""
    funding, out = default_funding, []
    for line in title.split("\n"):
        line = line.strip()
        m = COURSE.search(line)
        if m:
            out.append((int(m.group(1)), m.group(2).strip(), int(m.group(3)), funding))
        elif line:
            funding = line
    return out


def parse(page, d):
    """One row per school, course and funding type, with its number of batches (from the course cell's tooltip)."""
    page = re.sub(r"(?s)<!--.*?-->", "", page)
    rows = []
    for tr in re.findall(r"(?s)<tr[^>]*>(.*?)</tr>", page):
        cells = re.findall(r"(?s)<td([^>]*)>(.*?)</td>", tr)
        if len(cells) != 9 or not re.fullmatch(r"\d{4,5}", text(cells[0][1])):
            continue
        code, name_cell, management = text(cells[0][1]), cells[1][1], text(cells[2][1])
        m = re.search(r'title="([^"]*)"', cells[6][0])
        courses = tooltip_courses(html.unescape(m.group(1)) if m else "", management)
        listed = re.findall(r"\d+", re.sub(r"\[[^\]]*\]", " ", text(cells[6][1])))  # '1,11 [Aided] 39 [Un-Aided]'
        assert sorted(str(c) for c, *_ in courses) == sorted(listed), f"{code}: tooltip {courses} vs listed {listed}"
        local = [text(x) for x in re.split(r"<br\s*/?>", cells[3][1])]
        base = {"district_no": d, "district": DISTRICTS[d], "hss_code": code,
                "school_name": text(re.split(r"<br", name_cell)[0]),
                "pin": (re.search(r"Pin:\s*(\d{6})", text(name_cell)) or [None, None])[1],
                "management": management, "local_body": local[0], "taluk": local[1] if len(local) > 1 else None,
                "educational_district": text(cells[4][1]), "assembly": text(cells[5][1]), "gender": text(cells[7][1])}
        for c, cname, b, funding in courses:
            rows.append(base | {"course_code": c, "course_name": cname, "batches": b, "batch_funding": funding})
    return rows

## Run: 15 district pages

In [6]:
RAW.mkdir(parents=True, exist_ok=True)
session = requests.Session()
session.headers.update(HEADERS)
# passed with every request: a CURL_CA_BUNDLE variable (set by a PostgreSQL install) would override session.verify
CA = ca_bundle()
rows = []
for d in DISTRICTS:
    got = parse(fetch(d, session), d)
    rows += got
    print(f"{d:2d} {DISTRICTS[d]:20s} schools {len({r['hss_code'] for r in got}):4d} | batches {sum(r['batches'] for r in got):5d}")
courses = pd.DataFrame(rows)
assert not courses.duplicated(["hss_code", "course_code", "batch_funding"]).any()
OUT.parent.mkdir(parents=True, exist_ok=True)
courses.to_csv(OUT, index=False, encoding="utf-8")
print(f"written {OUT}: {courses['hss_code'].nunique():,} schools, {len(courses):,} school-course rows, "
      f"{courses['batches'].sum():,} batches")

 1 THIRUVANANTHAPURAM   schools  178 | batches   628


 2 KOLLAM               schools  138 | batches   532


 3 PATHANAMTHITTA       schools   95 | batches   296


 4 ALAPPUZHA            schools  124 | batches   453


 5 KOTTAYAM             schools  136 | batches   442


 6 IDUKKY               schools   82 | batches   237


... (5 outputs left out)

12 WAYANAD              schools   61 | batches   175


13 KANNUR               schools  161 | batches   556


14 KASARGODE            schools  106 | batches   286


15 OTHERS               schools   24 | batches    87
written C:\Users\ASUS\Desktop\sem3\BIG SPATIAL DATA\PROJECT- SPATIAL INEQUALITY ASESSMENT\data\interim\schools\hscap_hss_courses.csv: 2,100 schools, 5,936 school-course rows, 7,318 batches


## Checks

In [7]:
# Check 1: school counts per district against Sametham's own consolidation of HSE schools (Govt / Aided / Unaided)
f = RAW / "sametham_hse_consolidation.html"
if not f.exists():
    f.write_text(session.get(SAMETHAM_HSE, timeout=120, verify=CA).text, encoding="utf-8")
cells = [text(c) for c in re.findall(r"(?s)<td[^>]*>(.*?)</td>", f.read_text(encoding="utf-8"))]
ref = {}
for i, c in enumerate(cells):
    if re.fullmatch(r"\d{1,2}", c) and i + 6 < len(cells) and re.fullmatch(r"[A-Z ]+", cells[i + 1]):
        ref[int(c)] = dict(zip(["govt", "aided", "unaided", "technical", "total"], map(int, cells[i + 2:i + 7])))
schools = courses.drop_duplicates("hss_code")
mg = schools["management"].str.lower()
kind = pd.Series("other", index=schools.index).mask(mg.str.startswith("aided"), "aided")          .mask(mg.str.contains("government"), "govt").mask(mg.str.contains("un-aided"), "unaided")
cmp = pd.DataFrame({"hscap_govt": schools[kind.eq("govt")].groupby("district_no").size(),
                    "hscap_aided": schools[kind.eq("aided")].groupby("district_no").size(),
                    "hscap_unaided": schools[kind.eq("unaided")].groupby("district_no").size()}).fillna(0).astype(int)
cmp = cmp.join(pd.DataFrame(ref).T.add_prefix("sametham_"), how="outer")
print(cmp.to_string())
print("\nmanagement types:", schools["management"].value_counts().to_dict())
print("batches per course (how often):", courses["batches"].value_counts().sort_index().to_dict())
print("batches by funding type:", courses.groupby("batch_funding")["batches"].sum().to_dict())

             hscap_govt  hscap_aided  hscap_unaided  sametham_govt  sametham_aided  sametham_unaided  sametham_technical  sametham_total
district_no                                                                                                                             
1                    79           55             40           82.0            55.0              40.0                 2.0           179.0
2                    61           55             20           63.0            55.0              20.0                 0.0           138.0
3                    31           44             15           35.0            44.0              15.0                 2.0            96.0
4                    45           64             13           46.0            64.0              13.0                 2.0           125.0
5                    40           70             23           42.0            70.0              23.0                 1.0           136.0
6                    34           32     